In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# ============================================================
# Config
# ============================================================
BASE_DIR = Path("./")

INPUT_FILE = BASE_DIR / "event_representation_matrix_window0_scaled.csv"

OUT_DIR = BASE_DIR / "event_clustering_outputs"
OUT_DIR.mkdir(exist_ok=True)

N_CLUSTERS = 3
RANDOM_STATE = 42

# ============================================================
# 1. Load
# ============================================================
df = pd.read_csv(INPUT_FILE)
df["Date"] = pd.to_datetime(df["Date"])

feature_cols = [col for col in df.columns if col != "Date"]

X = df[feature_cols].values

print(f"[INFO] N events: {len(df)}")
print(f"[INFO] Features: {feature_cols}")

# ============================================================
# 2. KMeans Clustering
# ============================================================
kmeans = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=RANDOM_STATE,
    n_init=20
)

labels = kmeans.fit_predict(X)

df["Cluster"] = labels

# ============================================================
# 3. Cluster Centroid
# ============================================================
centroids = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=feature_cols
)
centroids["Cluster"] = range(N_CLUSTERS)

# ============================================================
# 4. PCA (2D 시각화용)
# ============================================================
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

df["PC1"] = X_pca[:, 0]
df["PC2"] = X_pca[:, 1]

# ============================================================
# 5. Save
# ============================================================
df.to_csv(OUT_DIR / "event_cluster_result.csv", index=False)
centroids.to_csv(OUT_DIR / "event_cluster_centroids.csv", index=False)

df[["Date", "Cluster", "PC1", "PC2"]].to_csv(
    OUT_DIR / "event_cluster_pca.csv", index=False
)

# ============================================================
# 6. Plot
# ============================================================
plt.figure(figsize=(8, 6))

for c in range(N_CLUSTERS):
    subset = df[df["Cluster"] == c]
    plt.scatter(subset["PC1"], subset["PC2"], label=f"Cluster {c}")

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Event Clustering (PCA)")
plt.legend()

plt.tight_layout()
plt.savefig(OUT_DIR / "event_cluster_pca.png", dpi=300)
plt.close()

print("[INFO] Clustering done")

[INFO] N events: 38
[INFO] Features: ['Magnitude', 'Top1_DeltaS', 'Top2_DeltaS', 'Top3_DeltaS', 'Peak', 'PeakHorizon', 'HalfLife', 'DurationAboveHalf', 'AUC']
[INFO] Clustering done
